In [1]:
#Solution 2

In [2]:
import pandas as pd
import torch
import os
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

In [3]:
# =========================
# CONFIG
# =========================
model_name = "google/gemma-2-9b-it"
input_file = "final_combined_dataset.csv"
output_file = "abstention_gemma_xml.csv"
save_every = 50
batch_size = 1

In [4]:
# =========================
# LOAD MODEL
# =========================
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3584, bias=False)
          (rotary_emb): Gemma2RotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=3584, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNorm((3584,), 

In [5]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv(input_file)

# Resume logic
if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    start_idx = len(df_existing)
    print(f" Resuming from row {start_idx}")
else:
    start_idx = 0

In [6]:
# =========================
# PROMPT (XML FORMAT)
# =========================
def build_prompt(row):
    return f"""
You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT, DO NOT DEVIATE):
<answer>a/b/c/d/idk</answer>
<reasoning>Give 2-3 line explanation</reasoning>

Only output these two tags. No extra text.
"""

In [7]:
# =========================
# PARSER (XML SAFE)
# =========================
def parse_output(output):
    final_answer = "idk"
    reasoning = ""

    try:
        # Extract answer
        ans_match = re.search(r"<answer>\s*(.*?)\s*</answer>", output, re.IGNORECASE)
        if ans_match:
            final_answer = ans_match.group(1).strip().lower()

        # Extract reasoning
        reason_match = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", output, re.IGNORECASE | re.DOTALL)
        if reason_match:
            reasoning = reason_match.group(1).strip()

    except:
        pass

    # fallback safety
    if final_answer not in ["a", "b", "c", "d", "idk"]:
        final_answer = "idk"

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning

In [8]:

# =========================
# INFERENCE LOOP
# =========================
results_buffer = []

for i in tqdm(range(start_idx, len(df), batch_size), desc="Running Inference"):

    batch = df.iloc[i:i+batch_size]
    prompts = [build_prompt(row) for _, row in batch.iterrows()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.0,
            do_sample=False
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for prompt, full_output in zip(prompts, decoded):
        cleaned = full_output[len(prompt):].strip()

        final_answer, reasoning = parse_output(cleaned)

        results_buffer.append({
            "pred_final_answer": final_answer,
            "pred_reasoning": reasoning
        })

    # =========================
    # SAVE EVERY 50 ROWS
    # =========================
    if (i + batch_size) % save_every == 0 or (i + batch_size) >= len(df):

        start_save_idx = i - len(results_buffer) + batch_size
        df_chunk = df.iloc[start_save_idx:i + batch_size].copy()

        df_chunk["pred_final_answer"] = [r["pred_final_answer"] for r in results_buffer]
        df_chunk["pred_reasoning"] = [r["pred_reasoning"] for r in results_buffer]

        if os.path.exists(output_file):
            df_chunk.to_csv(output_file, mode="a", index=False, header=False)
        else:
            df_chunk.to_csv(output_file, index=False)

        print(f" Saved up to row {i + batch_size}")

        results_buffer = []  # reset buffer

print(" Done! Fully robust pipeline completed.")

Running Inference:   0%|          | 0/3521 [00:00<?, ?it/s]/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Running Inference:   1%|▏         | 50/3521 [05:32<4:44:10,  4.91s/it] 

 Saved up to row 50


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Running Inference:   3%|▎         | 100/3521 [09:28<5:02:06,  5.30s/it]

 Saved up to row 100


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Running Inference:   4%|▍         | 150/3521 [13:32<3:33:00,  3.79s/it]/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Running Inference:   4%|▍         | 151/3521 [13:32<2:31:54,  2.70s/it]

...

 Saved up to row 3450


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Running Inference:  99%|█████████▉| 3500/3521 [3:50:54<01:09,  3.30s/it]

 Saved up to row 3500


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Running Inference: 100%|██████████| 3521/3521 [3:52:10<00:00,  3.96s/it]

 Saved up to row 3521
 Done! Fully robust pipeline completed.
